# AI Ethics, Fairness & Explainability (XAI)
**Level:** Professional | **Interactive Machine Learning Curriculum**

> Algorithmic bias, Demographic Parity, Equal Opportunity, Disparate Impact, SHAP (Shapley values), LIME, and Model Cards.

---

### 📖 Conceptual Foundations

#### The Imperative of Algorithmic Fairness

Machine learning models trained on historical data inevitably encode, amplify, and perpetuate historical societal biases. When deployed in high-stakes domains (credit lending, criminal justice recidivism, hiring, medical diagnosis), unfair algorithms inflict real-world harm on protected demographic groups (defined by race, gender, age, disability).


#### Core Mathematical Definitions of Fairness

| Fairness Criterion | Mathematical Formulation | Core Philosophy | Impossibility Trade-Off |
| --- | --- | --- | --- |
| Demographic Parity (Statistical Parity) | $P(\hat{Y}=1 \mid A=0) = P(\hat{Y}=1 \mid A=1)$ | Acceptance rates must be equal across protected groups, regardless of base rate differences. | Violates Individual Fairness if base qualification rates differ naturally. |
| Equal Opportunity (Hardt et al., 2016) | $P(\hat{Y}=1 \mid A=0, Y=1) = P(\hat{Y}=1 \mid A=1, Y=1)$ | True Positive Rates (TPR) must be equal across protected groups; qualified candidates have equal chances. | Requires knowing true labels $Y$; allows differing false positive rates. |
| Predictive Parity (Equal PPV) | $P(Y=1 \mid A=0, \hat{Y}=1) = P(Y=1 \mid A=1, \hat{Y}=1)$ | Precision (Positive Predictive Value) must be equal across protected groups. | **Chouldechova / Kleinberg Impossibility Theorem:** DP, Equal Opportunity, and Predictive Parity cannot all be satisfied simultaneously unless base rates are identical! |


#### SHAP: Shapley Additive Explanations (Lundberg & Lee, 2017)

Rooted in cooperative game theory (Lloyd Shapley, Nobel Memorial Prize 1953), **SHAP** calculates the marginal contribution $\phi_i$ of each feature $i$ across all possible feature subsets $S \subseteq F \setminus \{i\}$:

$$\phi_i(x) = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(|F| - |S| - 1)!}{|F|!} \left[ f_x(S \cup \{i\}) - f_x(S) \right]$$

- **Efficiency:** The sum of Shapley values equals the difference between model prediction and base value: $f(x) - \mathbb{E}[f(x)] = \sum_i \phi_i$.
- **Fairness:** It is the *unique* attribution method satisfying Local Accuracy, Missingness, and Consistency.


#### LIME: Local Interpretable Model-agnostic Explanations (Ribeiro et al., 2016)

While complex models (deep nets, gradient boosted trees) are globally non-linear and uninterpretable, their decision boundary is locally smooth around any specific instance $x$. **LIME** fits an interpretable, sparse linear surrogate model $g \in G$ weighted by proximity $\pi_x(z)$:

$$\arg\min_{g \in G} \mathcal{L}(f, g, \pi_x) + \Omega(g)$$

Explains *why* the black-box model made that specific prediction for that individual sample.



In [ ]:
# %matplotlib inline
import numpy as np
import itertools

# Simulated black-box model predicting loan score based on [Income, CreditScore, Age]
def black_box_model(features):
    # features: boolean presence of features [inc, cred, age]
    val = 500 # base value
    if features[0]: val += 120 # Income
    if features[1]: val += 200 # CreditScore
    if features[2]: val += 40  # Age
    if features[0] and features[1]: val += 50 # Non-linear synergy between Income & Credit
    return val

# Calculate exact Shapley values for 3 features
feature_names = ['Income', 'CreditScore', 'Age']
n = len(feature_names)
shap_values = np.zeros(n)

for i in range(n):
    marginal_contributions = []
    weights = []
    other_indices = [idx for idx in range(n) if idx != i]
    
    for subset_len in range(n):
        for subset in itertools.combinations(other_indices, subset_len):
            # Weight for this coalition size
            weight = (np.math.factorial(len(subset)) * np.math.factorial(n - len(subset) - 1)) / np.math.factorial(n)
            
            # Evaluate with and without feature i
            mask_without = [False] * n
            for idx in subset: mask_without[idx] = True
            
            mask_with = list(mask_without)
            mask_with[i] = True
            
            contribution = black_box_model(mask_with) - black_box_model(mask_without)
            shap_values[i] += weight * contribution

base_val = black_box_model([False, False, False])
total_val = black_box_model([True, True, True])

print(f'Base Value (E[f(x)]): {base_val}')
print(f'Total Prediction f(x): {total_val}')
print(f'Total Difference: {total_val - base_val}\n')
for name, sv in zip(feature_names, shap_values):
    print(f'Feature: {name:<12} -> SHAP Value: {sv:+.1f}')
print(f'Sum of SHAP Values: {np.sum(shap_values):.1f} (Efficiency axiom satisfied!)')


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

